# 01. Chuẩn bị dữ liệu Chương 2 — Anh

SBA 7(a), FY2000–FY2019, bản chụp 30/06/2026; FRED-MD 09/2026. Chạy sau khi tải dữ liệu theo README. Không sửa CSV gốc. Trên Colab, upload cả thư mục và mount Drive trước. Nếu thiếu thư viện, chạy `%pip install pandas numpy pyarrow openpyxl requests`.

Nguồn: [SBA](https://data.sba.gov/dataset/7a-504-foia); [FRED-MD](https://www.stlouisfed.org/research/economists/mccracken/fred-databases).

In [1]:

from pathlib import Path
import sys, json, hashlib, warnings
import numpy as np
import pandas as pd
from IPython.display import display, Markdown

# Trên Colab: mount Drive trước, rồi điền BASE là thư mục PTDLNC đã tải lên.
# Không cần sửa khi mở notebook từ bên trong thư mục dự án trên máy cá nhân.
BASE = next((p for p in [Path.cwd(), *Path.cwd().parents, Path.cwd() / 'PTDLNC']
             if (p / 'data' / 'raw').is_dir()), None)
if BASE is None:
    raise FileNotFoundError('Đặt BASE tới thư mục PTDLNC có data/raw (xem README).')
RANDOM_STATE = 42
RAW, PROCESSED, META = [BASE / 'data' / x for x in ('raw', 'processed', 'metadata')]
DATASET = json.loads((META / 'dataset_config.json').read_text(encoding='utf-8'))
SNAPSHOT = pd.Timestamp(DATASET['snapshot'])
SNAPSHOT_VI = DATASET['snapshot_vi']
WORK = BASE / 'bai_lam_Anh_Chuong2'
FIGURES, TABLES = WORK / 'figures', WORK / 'tables'
for p in (PROCESSED, META, FIGURES, TABLES): p.mkdir(parents=True, exist_ok=True)
def save_json(obj, path):
    path.write_text(json.dumps(obj, ensure_ascii=False, indent=2, default=str), encoding='utf-8')
def sha256(path):
    h = hashlib.sha256()
    with path.open('rb') as f:
        for b in iter(lambda: f.read(1024*1024), b''): h.update(b)
    return h.hexdigest()
def table(frame, name):
    frame.to_csv(TABLES / (name + '.csv'), index=False, encoding='utf-8-sig', float_format='%.12g')
    display(frame.head(30))


## 1. Kiểm tra cấu trúc thực tế, bảo vệ định danh và tạo nhãn
Bản 260630 dùng tên cột CamelCase, ngày ISO YYYY-MM-DD và trạng thái trả đủ `P I F`. `ApprovalFY` ánh xạ thành `ApprovalFiscalYear`; `P I F` chuẩn hóa thành `PIF` trước khi tạo nhãn. CSV không còn Subprogram: không tạo giả cột này; dùng ProcessingMethod. Cache chỉ dùng lại khi snapshot và SHA-256 từng CSV khớp; khác nguồn sẽ xây lại Parquet. Từ điển không cung cấp khóa duy nhất của khoản vay; LocationID là mã ngân hàng. Chỉ đếm fingerprint lặp và không xóa hàng. Đọc các trường gốc trong bộ nhớ theo khối chỉ để kiểm tra fingerprint, sau đó loại định danh ngay; không hiển thị hàng thô.

In [2]:

files = [RAW / item['file'] for item in DATASET['sources']]
assert all(f.exists() for f in files), 'Thiếu CSV SBA đúng phiên bản trong dataset_config.json.'
dictionary_path = RAW / '7a_504_foia_data_dictionary.xlsx'
dictionary = pd.read_excel(dictionary_path, sheet_name='7(a) Data Dictionary')
definitions = dict(dictionary.iloc[:, :2].itertuples(index=False, name=None))
canonical = {str(k).lower(): str(k) for k in definitions}
canonical.update(subprogram='Subprogram', approvalfy='ApprovalFiscalYear')
KEEP = ['AsOfDate','Program','BorrState','BankName','GrossApproval','SBAGuaranteedApproval',
        'ApprovalDate','ApprovalFiscalYear','FirstDisbursementDate','ProcessingMethod',
        'InitialInterestRate','FixedorVariableInterestInd','TermInMonths','NaicsCode','NaicsDescription',
        'FranchiseCode','BusinessType','BusinessAge','LoanStatus','PaidInFullDate','ChargeOffDate',
        'GrossChargeOffAmount','JobsSupported','ProjectState','RevolverStatus','CollateralInd','SoldSecMrktInd']
NUMERIC = ['GrossApproval','SBAGuaranteedApproval','ApprovalFiscalYear','InitialInterestRate',
           'TermInMonths','GrossChargeOffAmount','JobsSupported']
DATES = ['AsOfDate','ApprovalDate','FirstDisbursementDate','PaidInFullDate','ChargeOffDate']
DENIED = {'BorrName','BorrStreet','BorrCity','BorrZip','LocationID','BankStreet','BankZip',
          'BankFDICNumber','BankNCUANumber','FranchiseName','ProjectCounty','CongressionalDistrict'}
fingerprints = set()
audits, parts, invalid = [], [], {}
existing = PROCESSED / 'loans_clean.parquet'
source_hashes = {f.name:sha256(f) for f in files}
previous = json.loads((META/'pipeline_audit.json').read_text(encoding='utf-8')) if (META/'pipeline_audit.json').exists() else {}
cache_matches = (existing.exists() and previous.get('snapshot') == DATASET['snapshot'] and
                 {r['file']:r['sha256'] for r in previous.get('files',[])} == source_hashes)
rebuilt = not cache_matches
if cache_matches:
    loans = pd.read_parquet(existing)
    assert not DENIED.intersection(loans.columns), 'Parquet hiện có chứa định danh.'
    assert {'fy','approval_month','target','is_resolved',*KEEP}.issubset(loans.columns), 'Parquet chưa đủ hợp đồng EDA.'
    assert loans.AsOfDate.eq(SNAPSHOT).all(), 'Cache khác ngày chụp được cấu hình.'
    print('Sử dụng Parquet có sẵn, không ghi đè:', len(loans))
else:
    for f, spec in zip(files, DATASET['sources']):
        lo,hi = spec['fy_start'],spec['fy_end']
        # Kiểm tra UTF-8 toàn bộ byte, không suy đoán encoding từ vài dòng đầu.
        import codecs
        decoder = codecs.getincrementaldecoder('utf-8-sig')('strict')
        try:
            with f.open('rb') as source:
                for block in iter(lambda: source.read(1024*1024), b''): decoder.decode(block)
                decoder.decode(b'', final=True)
            encoding = 'utf-8-sig'
        except UnicodeDecodeError:
            encoding = 'cp1252'
        n, repeated, normalized = 0, 0, 0
        header = pd.read_csv(f, nrows=0, encoding=encoding).columns.tolist()
        missing = set(KEEP) - {canonical.get(c.lower(),c) for c in header}
        assert not missing, f'Thiếu cột: {missing}'
        selected = KEEP + (['Subprogram'] if any(c.lower()=='subprogram' for c in header) else [])
        raw_status_counts = {}
        for chunk in pd.read_csv(f, dtype='string', encoding=encoding, chunksize=100_000, keep_default_na=False):
            n += len(chunk)
            # Fingerprint toàn hàng gốc chỉ phục vụ kiểm tra, không xuất và không xóa hàng.
            # Không có mã khoản vay duy nhất: đây không phải xác nhận trùng khoản vay.
            h = pd.util.hash_pandas_object(chunk, index=False).to_numpy()
            before = len(fingerprints)
            fingerprints.update(map(int,h))
            repeated += len(chunk) - (len(fingerprints)-before)
            chunk = chunk.rename(columns={c:canonical.get(c.lower(),c) for c in chunk.columns})[selected].copy()
            for c in chunk:
                chunk[c] = chunk[c].str.strip().replace('', pd.NA)
            for c in NUMERIC:
                parsed = pd.to_numeric(chunk[c], errors='coerce')
                invalid[c] = invalid.get(c,0) + int((chunk[c].notna() & parsed.isna()).sum())
                chunk[c] = parsed.astype('float64')
            for c in DATES:
                iso = chunk[c].str.match(r'^\d{4}-\d{2}-\d{2}$',na=False)
                parsed = pd.to_datetime(chunk[c].where(iso), format='%Y-%m-%d', errors='coerce')
                us_dates = pd.to_datetime(chunk[c].where(~iso), format='%m/%d/%Y', errors='coerce')
                parsed = parsed.fillna(us_dates)
                invalid[c] = invalid.get(c,0) + int((chunk[c].notna() & parsed.isna()).sum())
                chunk[c] = parsed
            original_status = chunk['LoanStatus']
            for code,count in original_status.value_counts(dropna=False).items():
                raw_status_counts[str(code)] = raw_status_counts.get(str(code),0)+int(count)
            chunk['LoanStatus'] = original_status.str.upper().str.replace(r'\s+', '', regex=True)
            normalized += int((original_status != chunk['LoanStatus']).fillna(False).sum())
            assert chunk.AsOfDate.eq(SNAPSHOT).all(), 'Sai ngày chụp SBA!'
            assert chunk.ApprovalFiscalYear.between(lo,hi).all(), 'FY ngoài phạm vi file!'
            chunk['source_file'] = f.name
            parts.append(chunk)
        audits.append(dict(file=f.name,rows=n,raw_columns=len(header),encoding=encoding,
                           sha256=sha256(f),raw_header=header,repeated_row_fingerprints=repeated,
                           status_normalized_rows=normalized,raw_status_counts=raw_status_counts,
                           date_formats=['ISO YYYY-MM-DD','US MM/DD/YYYY'],
                           unavailable_columns=['Subprogram'] if 'Subprogram' not in selected else [],
                           unique_loan_key='Not supplied; no deduplication'))
        print(f.name, n, 'hàng; encoding',encoding,'; fingerprint lặp:',repeated)
    loans = pd.concat(parts, ignore_index=True)
    del parts, fingerprints
    loans['fy'] = loans.ApprovalFiscalYear.astype('Int16')
    loans['approval_month'] = loans.ApprovalDate.dt.to_period('M').dt.to_timestamp()
    loans['is_resolved'] = loans.LoanStatus.isin(['CHGOFF','PIF'])
    loans['target'] = loans.LoanStatus.map({'CHGOFF':1,'PIF':0}).astype('Int8')
    assert not DENIED.intersection(loans.columns)
    assert loans.ApprovalDate.notna().all()
    assert loans.fy.eq((loans.ApprovalDate.dt.year + loans.ApprovalDate.dt.month.ge(10)).astype('Int16')).all()
    assert loans.LoanStatus.isin(['PIF','CHGOFF','CANCLD','EXEMPT','COMMIT']).all()
    assert loans.is_resolved.eq(loans.target.notna()).all()
    assert not any(invalid.values()), f'Lỗi chuyển kiểu cần kiểm tra: {invalid}'
    temporary = existing.with_suffix('.tmp.parquet')
    loans.to_parquet(temporary, index=False)
    temporary.replace(existing)
    save_json({'files':audits,'parse_failures':invalid,'rows':len(loans),
               'snapshot':DATASET['snapshot'],'source_hashes':source_hashes,
               'cache_policy':'Reuse only when configured snapshot and raw SHA-256 match.',
               'date_start':loans.ApprovalDate.min(),'date_end':loans.ApprovalDate.max(),
               'columns':loans.columns.tolist(),'duplicate_policy':'No loan ID; retain every row.',
               'dictionary_version':'Current official dictionary downloaded; no historical vintage claimed.'}, META/'pipeline_audit.json')
assert loans.fy.between(2000,2019).all()
assert loans.target.dropna().isin([0,1]).all()
assert loans.loc[~loans.is_resolved,'target'].isna().all()
assert loans.loc[loans.LoanStatus.eq('CHGOFF'),'target'].eq(1).all()
assert loans.loc[loans.LoanStatus.eq('PIF'),'target'].eq(0).all()
assert not DENIED.intersection(loans.columns)
table(loans.groupby('LoanStatus',dropna=False).size().rename('n').reset_index(), 'B2_trang_thai')


Sử dụng Parquet có sẵn, không ghi đè: 1236084


,LoanStatus,n
0,CANCLD,149020
1,CHGOFF,178504
2,COMMIT,44
3,EXEMPT,53573
4,PIF,854943


## 2. FRED-MD và hợp đồng bàn giao
Tách dòng Transform trước khi chuyển số, xác nhận chuỗi tháng liên tục và checksum của Fred.csv khớp bản chính thức. Giữ nguyên các trạng thái không có nhãn trong dữ liệu mô tả. Đầu ra: ba Parquet và metadata; không ghi đè dữ liệu nhóm có sẵn.

In [3]:

official, local = RAW/'FRED_MD_2026-09.csv', RAW/'Fred_original.csv'
assert sha256(official) == sha256(local), 'Fred.csv không khớp byte với bản chính thức 09/2026.'
fred_raw = pd.read_csv(official)
assert fred_raw.iloc[0,0] == 'Transform:'
tcodes = pd.to_numeric(fred_raw.iloc[0,1:], errors='raise').astype(int)
tcodes.rename('tcode').rename_axis('variable').reset_index().to_csv(META/'fred_tcodes.csv',index=False)
fred = fred_raw.iloc[1:].copy()
fred['sasdate'] = pd.to_datetime(fred.sasdate,format='%m/%d/%Y',errors='raise')
fred = fred.set_index('sasdate').apply(pd.to_numeric,errors='raise').sort_index()
assert fred.index.is_unique and fred.index.is_monotonic_increasing
assert fred.index.equals(pd.date_range(fred.index.min(),fred.index.max(),freq='MS',name='sasdate'))
fred_path=PROCESSED/'fred_md_monthly.parquet'
if not fred_path.exists(): fred.to_parquet(fred_path)
else:
    supplied=pd.read_parquet(fred_path)
    pd.testing.assert_frame_equal(supplied,fred,check_dtype=False,check_freq=False)
monthly = loans.groupby('approval_month').agg(n_loans=('fy','size'),gross_approval=('GrossApproval','sum')).sort_index()
monthly_path=PROCESSED/'monthly_series.parquet'
if rebuilt or not monthly_path.exists(): monthly.to_parquet(monthly_path)
else: pd.testing.assert_frame_equal(pd.read_parquet(monthly_path),monthly,check_dtype=False,check_freq=False)
assert len(monthly)==240 and monthly.n_loans.sum()==len(loans)
source_profile = {'observations':len(fred),'macro_variables':len(fred.columns),
                  'first_month':str(fred.index.min().date()),'last_month':str(fred.index.max().date()),
                  'sha256':sha256(official),'local_matches_official':True,
                  'monthly_loan_observations':len(monthly),'loan_rows':len(loans)}
save_json(source_profile,META/'fred_profile.json')
table(pd.DataFrame([source_profile]),'B1_fred_nguon')
display(Markdown('**Hoàn tất:** dữ liệu gốc được giữ nguyên; ba Parquet không có định danh người vay. Không có thao tác bù thiếu, cắt ngoại lai hoặc huấn luyện mô hình.'))


,observations,macro_variables,first_month,last_month,sha256,local_matches_official,monthly_loan_observations,loan_rows
0,812,126,1959-01-01,2026-08-01,d6e239038afbdbf4667c2a63c13c942817bc196fedab43...,True,240,1236084


**Hoàn tất:** dữ liệu gốc được giữ nguyên; ba Parquet không có định danh người vay. Không có thao tác bù thiếu, cắt ngoại lai hoặc huấn luyện mô hình.